Qwen3.5-0.8B (1K select), w. LoRA

In [ ]:
# 1️⃣ Install dependencies (run in Colab / local GPU)
!pip install --upgrade -qqq uv
!uv pip install -qqq torch==2.8.0 torchvision bitsandbytes xformers==0.0.32.post2 \
    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
    "unsloth[base] @ git+https://github.com/unslothai/unsloth"
!uv pip install --upgrade --no-deps tokenizers trl==0.22.2 transformers==5.2.0
!uv pip install --no-build-isolation flash-linear-attention causal_conv1d==1.6.0
!pip install pillow datasets

Using Python 3.12.13 environment at: /usr
Resolved 3 packages in 75ms
Prepared 1 package in 0.33ms
Uninstalled 1 package in 66ms
Installed 1 package in 128ms
 - transformers==5.3.0
 + transformers==5.2.0
Using Python 3.12.13 environment at: /usr
Checked 2 packages in 90ms


In [ ]:

import torch
from unsloth import FastVisionModel
from datasets import load_dataset
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# 1. Load with 4bit to save VRAM, but use Float32 for compute stability on T4
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model.safetensors-00001-of-00001.safeten(…):   0%|          | 0.00/1.75G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/336 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/20.0M [00:00<?, ?B/s]

video_preprocessor_config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

In [ ]:
# 2. Apply LoRA
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=True,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    finetune_mlp_modules=True,
    r=16,
    lora_alpha=16,
    random_state=3407,
)

Unsloth: Making `model.base_model.model.model.visual` require gradients


In [ ]:
# 3. CRITICAL FIX FOR T4:
# We must ensure NO BFloat16 tensors exist.
FastVisionModel.for_training(model)
model = model.to(torch.float32) # Force base to Float32


In [ ]:
# 4. Dataset Prep (Same as before)
dataset = load_dataset("SimulaMet/Kvasir-VQA-x1", split="train").select(range(1000))
def formatting_prompts_func(examples):
    conversations = []
    for q, a, img in zip(examples["question"], examples["answer"], examples["image"]):
        conv = [
            {"role": "user", "content": [{"type": "text", "text": q}, {"type": "image", "image": img}]},
            {"role": "assistant", "content": [{"type": "text", "text": a}]}
        ]
        conversations.append(conv)
    return {"messages": conversations}
dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/1.65M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/143594 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/15955 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [ ]:
# 5. Trainer with NO mixed precision
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=dataset,
    max_seq_length=1024, # Reduced for T4 Float32 memory load
    args=SFTConfig(
        per_device_train_batch_size=1, # Reduced for stability
        gradient_accumulation_steps=8,
        warmup_steps=5,
        max_steps=30,
        learning_rate=2e-4,
        fp16=False,            # Disable to stop 'Half' mismatch
        bf16=False,            # Disable to stop 'BFloat16' mismatch
        optim="adamw_8bit",
        logging_steps=1,
        output_dir="outputs",
        remove_unused_columns=False,
        dataset_kwargs={"skip_prepare_dataset": True}
    ),
)


Unsloth: Model does not have a default image size - using 512
Unsloth: Switching to float32 training since model cannot work with float16


In [ ]:
# 6. Final check: Force all parameters to float32 right before training
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 30
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 13,181,952 of 866,167,872 (1.52% trained)


Step,Training Loss
1,2.859256
2,2.624732
3,2.677584
4,2.255519
5,1.917692
6,2.049402
7,1.938486
8,1.585985
9,1.510417
10,1.153398


TrainOutput(global_step=30, training_loss=1.3872521142164866, metrics={'train_runtime': 703.0383, 'train_samples_per_second': 0.341, 'train_steps_per_second': 0.043, 'total_flos': 222571309056000.0, 'train_loss': 1.3872521142164866, 'epoch': 0.24})

In [ ]:
# Save the LoRA adapters first (lightweight)
model.save_pretrained("qwen3_5_kvasir_lora")
tokenizer.save_pretrained("qwen3_5_kvasir_lora")

['qwen3_5_kvasir_lora/processor_config.json']

In [ ]:
# Merge LoRA into the main model and save as 16-bit (Float16)
# This is much faster for future inference
model.save_pretrained_merged("qwen3_5_kvasir_merged", tokenizer, save_method = "merged_16bit")


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...



Unsloth: Copying 1 files from cache to `qwen3_5_kvasir_merged`:   0%|          | 0/1 [00:00<?, ?it/s]
Unsloth: Copying 1 files from cache to `qwen3_5_kvasir_merged`: 100%|██████████| 1/1 [00:23<00:00, 23.58s/it]


Successfully copied all 1 files from cache to `qwen3_5_kvasir_merged`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 1/1 [00:00<00:00, 8355.19it/s]

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:00<00:00, 60.62s/it]


Unsloth: Merge process complete. Saved to `/content/qwen3_5_kvasir_merged`


In [ ]:
from PIL import Image
from transformers import TextStreamer

from unsloth import FastVisionModel

# Load the merged model
model, tokenizer = FastVisionModel.from_pretrained(
    "qwen3_5_kvasir_merged",
    load_in_4bit = True,
)
FastVisionModel.for_inference(model)

# 1. Load a test sample (one the model hasn't seen, or the first one)
test_sample = dataset[0]
# If 'dataset' was mapped, 'messages' contains the data.
# Let's pull the first user message:
image = test_sample["messages"][0]["content"][1]["image"]
question = test_sample["messages"][0]["content"][0]["text"]

# 2. Prepare for Inference
FastVisionModel.for_inference(model)

# 3. Format the prompt
instruction = [
    {"role": "user", "content": [
        #{"type": "text", "text": f"Analyze this endoscopy image: {question}"},
        {"type": "text", "text": f"{question}"},
        {"type": "image", "image": image}
    ]}
]

input_text = tokenizer.apply_chat_template(instruction, add_generation_prompt=True)
inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

# 4. Generate with Streaming
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print(f"\n--- Question: {question} ---")
print("--- Model Answer: ---")
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.1, # Low temperature for medical accuracy
)

==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tokenizer you are loading from 'qwen3_5_kvasir_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



--- Question: Are there any abnormalities, polyps, or anatomical landmarks visible in the image? ---
--- Model Answer: ---
No evidence of polyps or anatomical landmarks observed<|im_end|>


Qwen3.5-0.8B (0.2K select) classic

In [ ]:
!pip install -q transformers accelerate datasets evaluate peft sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.3 MB/s eta 0:00:00


In [ ]:
!pip install transformers==4.40.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.0/138.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 95.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 125.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.2.0
    Uninstalling transformers-5.2.0:
      Successfully uninstalled transformers-5.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follo

In [ ]:
from unsloth import FastVisionModel
import torch
from datasets import load_dataset
import evaluate
import time
from tqdm import tqdm

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
model, processor = FastVisionModel.from_pretrained(
    model_name = "unsloth/Qwen3.5-0.8B",
    load_in_4bit = True,   # IMPORTANT pentru Colab
)

FastVisionModel.for_inference(model)

==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

Qwen3_5ForConditionalGeneration(
  (model): Qwen3_5Model(
    (visual): Qwen3_5VisionModel(
      (patch_embed): Qwen3_5VisionPatchEmbed(
        (proj): Conv3d(3, 768, kernel_size=(2, 16, 16), stride=(2, 16, 16))
      )
      (pos_embed): Embedding(2304, 768)
      (rotary_pos_emb): Qwen3_5VisionRotaryEmbedding()
      (blocks): ModuleList(
        (0-11): 12 x Qwen3_5VisionBlock(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Qwen3_5VisionAttention(
            (qkv): Linear4bit(in_features=768, out_features=2304, bias=True)
            (proj): Linear4bit(in_features=768, out_features=768, bias=True)
          )
          (mlp): Qwen3_5VisionMLP(
            (linear_fc1): Linear4bit(in_features=768, out_features=3072, bias=True)
            (linear_fc2): Linear4bit(in_features=3072, out_features=768, bias=True)
            (act_fn): GELUTanh()
          )
        )
  

In [ ]:
dataset = load_dataset("SimulaMet/Kvasir-VQA-x1", split="test")
dataset = dataset.select(range(200))  # test rapid

In [ ]:
bleu = evaluate.load("bleu")
rouge = evaluate.load("rouge")

In [ ]:
!pip install rouge_score

  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=f3b11ff03b2bdc51a3d976cf50bed167d9b3c04becab94973c81842f75f8aff8
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
def generate_answer(image, question):
    prompt = f"""You are a medical VQA assistant.
Answer the question based only on the image.

Question: {question}
Answer:"""

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False
        )

    answer = processor.batch_decode(outputs, skip_special_tokens=True)[0]

    if "Answer:" in answer:
        answer = answer.split("Answer:")[-1].strip()

    return answer

In [ ]:
import re

def normalize(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
# 4. Dataset Prep (Same as before)
dataset = load_dataset("SimulaMet/Kvasir-VQA-x1", split="train").select(range(1000))
def formatting_prompts_func(examples):
    conversations = []
    for q, a, img in zip(examples["question"], examples["answer"], examples["image"]):
        conv = [
            {"role": "user", "content": [{"type": "text", "text": q}, {"type": "image", "image": img}]},
            {"role": "assistant", "content": [{"type": "text", "text": a}]}
        ]
        conversations.append(conv)
    return {"messages": conversations}
dataset = dataset.map(formatting_prompts_func, batched=True, remove_columns=dataset.column_names)

In [ ]:
from PIL import Image
from transformers import TextStreamer

from unsloth import FastVisionModel

# Load the merged model
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-0.8B",
    load_in_4bit = True,
)
FastVisionModel.for_inference(model)

# 1. Load a test sample (one the model hasn't seen, or the first one)
test_sample = dataset[0]
# If 'dataset' was mapped, 'messages' contains the data.
# Let's pull the first user message:
image = test_sample["messages"][0]["content"][1]["image"]
question = test_sample["messages"][0]["content"][0]["text"]

# 2. Prepare for Inference
FastVisionModel.for_inference(model)

# 3. Format the prompt
instruction = [
    {"role": "user", "content": [
        #{"type": "text", "text": f"Analyze this endoscopy image: {question}"},
        {"type": "text", "text": f"{question}"},
        {"type": "image", "image": image}
    ]}
]

input_text = tokenizer.apply_chat_template(instruction, add_generation_prompt=True)
inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

# 4. Generate with Streaming
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print(f"\n--- Question: {question} ---")
print("--- Model Answer: ---")
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.1, # Low temperature for medical accuracy
)

==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]


--- Question: Are there any abnormalities, polyps, or anatomical landmarks visible in the image? ---
--- Model Answer: ---
Based on the visual evidence provided in the image, there are no obvious abnormalities, polyps, or distinct anatomical landmarks visible.

The image appears to be a close-up endoscopic view of the esophagus or stomach lining. The mucosa (the inner lining) appears relatively healthy with a pinkish, smooth surface.

Key features that suggest normal appearance include:
*   **Surface Texture:** The tissue has a consistent, smooth, and slightly shiny appearance.
*   **Bubbles:** There are numerous small bubbles (likely from the introduction of a solution or air) scattered across the field of view, which is a


Bleu score

In [ ]:
predictions = ["polyp", "no polyp", "ulcer"]

references = [
    ["polyp"],
    ["no polyp"],
    ["ulcer"]
]

In [ ]:
import evaluate

bleu = evaluate.load("bleu")

bleu_score = bleu.compute(
    predictions=predictions,
    references=references  # must be list of lists
)

print("BLEU score:", bleu_score["bleu"])

BLEU score: 0.0


In [ ]:
bleu_score = bleu.compute(
    predictions=predictions,
    references=references,
    max_order=1
)

print("BLEU-1:", bleu_score["bleu"])

BLEU-1: 1.0


In [ ]:
clean_predictions = []
clean_references = []

for pred, ref in zip(predictions, references):
    if pred is None:
        continue
    if ref is None:
        continue

    pred = str(pred).strip()

    # ref must be a non-empty list of non-empty strings
    if isinstance(ref, list):
        ref = [str(r).strip() for r in ref if str(r).strip() != ""]
    else:
        ref = [str(ref).strip()] if str(ref).strip() != "" else []

    if pred != "" and len(ref) > 0:
        clean_predictions.append(pred)
        clean_references.append(ref)

print("Original predictions:", len(predictions))
print("Original references :", len(references))
print("Clean predictions   :", len(clean_predictions))
print("Clean references    :", len(clean_references))
print("Example prediction  :", clean_predictions[0] if len(clean_predictions) else "NONE")
print("Example reference   :", clean_references[0] if len(clean_references) else "NONE")

Original predictions: 3
Original references : 3
Clean predictions   : 3
Clean references    : 3
Example prediction  : polyp
Example reference   : ['polyp']


In [ ]:
import evaluate

bleu = evaluate.load("bleu")

bleu_score = bleu.compute(
    predictions=clean_predictions,
    references=clean_references
)

print("BLEU score:", bleu_score["bleu"])
print(bleu_score)

BLEU score: 0.0
{'bleu': 0.0, 'precisions': [1.0, 1.0, 0.0, 0.0], 'brevity_penalty': 1.0, 'length_ratio': 1.0, 'translation_length': 4, 'reference_length': 4}


In [ ]:
for i, (pred, ref) in enumerate(zip(predictions, references)):
    bad = False

    if pred is None or str(pred).strip() == "":
        bad = True

    if ref is None:
        bad = True
    elif isinstance(ref, list):
        if len(ref) == 0 or all(str(r).strip() == "" for r in ref):
            bad = True
    else:
        if str(ref).strip() == "":
            bad = True

    if bad:
        print(f"Bad sample at index {i}")
        print("Prediction:", pred)
        print("Reference :", ref)
        print("-" * 40)

In [ ]:
predictions.append(str(pred).strip())
references.append([str(gt).strip()])

Evaluation

In [ ]:
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import evaluate

In [ ]:
smooth_fn = SmoothingFunction().method1

def mean_sentence_bleu(predictions, references):
    scores = []

    for pred, ref_list in zip(predictions, references):
        pred_tokens = pred.split()
        ref_tokens = [ref.split() for ref in ref_list]

        score = sentence_bleu(
            ref_tokens,
            pred_tokens,
            smoothing_function=smooth_fn
        )
        scores.append(score)

    return float(np.mean(scores)), scores

In [ ]:
bleu_metric = evaluate.load("bleu")

def corpus_bleu(predictions, references):
    result = bleu_metric.compute(
        predictions=predictions,
        references=references,
        smooth=True
    )
    return result["bleu"], result

In [ ]:
predictions_qwen = [
    "a polyp",
    "no polyp",
    "ulcer",
    "normal mucosa",
    "inflammation"
]

references_qwen = [
    ["a polyp"],
    ["no polyp"],
    ["ulcer"],
    ["normal mucosa"],
    ["inflammation"]
]

In [ ]:
qwen_mean_bleu, qwen_bleu_list = mean_sentence_bleu(predictions_qwen, references_qwen)
qwen_corpus_bleu, qwen_corpus_details = corpus_bleu(predictions_qwen, references_qwen)

print("Qwen mean sentence BLEU:", qwen_mean_bleu)
print("Qwen corpus BLEU:", qwen_corpus_bleu)

Qwen mean sentence BLEU: 0.26086783601165975
Qwen corpus BLEU: 1.0


Qwen LoRA

In [ ]:
from PIL import Image
from transformers import TextStreamer

from unsloth import FastVisionModel

# Load the merged model
model, tokenizer = FastVisionModel.from_pretrained(
    "qwen3_5_kvasir_merged",
    load_in_4bit = True,
)
FastVisionModel.for_inference(model)

# 1. Load a test sample (one the model hasn't seen, or the first one)
test_sample = dataset[0]
# If 'dataset' was mapped, 'messages' contains the data.
# Let's pull the first user message:
image = test_sample["messages"][0]["content"][1]["image"]
question = test_sample["messages"][0]["content"][0]["text"]

# 2. Prepare for Inference
FastVisionModel.for_inference(model)

# 3. Format the prompt
instruction = [
    {"role": "user", "content": [
        #{"type": "text", "text": f"Analyze this endoscopy image: {question}"},
        {"type": "text", "text": f"{question}"},
        {"type": "image", "image": image}
    ]}
]

input_text = tokenizer.apply_chat_template(instruction, add_generation_prompt=True)
inputs = tokenizer(image, input_text, add_special_tokens=False, return_tensors="pt").to("cuda")

# 4. Generate with Streaming
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print(f"\n--- Question: {question} ---")
print("--- Model Answer: ---")
_ = model.generate(
    **inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=0.1, # Low temperature for medical accuracy
)


==((====))==  Unsloth 2026.3.17: Fast Qwen3_5 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.32.post2. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for qwen3_5 won't work! Using float32.


Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tokenizer you are loading from 'qwen3_5_kvasir_merged' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.



--- Question: Are there any abnormalities, polyps, or anatomical landmarks visible in the image? ---
--- Model Answer: ---
No evidence of polyps or anatomical landmarks observed<|im_end|>


In [ ]:
smooth_fn = SmoothingFunction().method1

def mean_sentence_bleu(predictions, references):
    scores = []

    for pred, ref_list in zip(predictions, references):
        pred_tokens = pred.split()
        ref_tokens = [ref.split() for ref in ref_list]

        score = sentence_bleu(
            ref_tokens,
            pred_tokens,
            smoothing_function=smooth_fn
        )
        scores.append(score)

    return float(np.mean(scores)), scores

In [ ]:
bleu_metric = evaluate.load("bleu")

def corpus_bleu(predictions, references):
    result = bleu_metric.compute(
        predictions=predictions,
        references=references,
        smooth=True
    )
    return result["bleu"], result

In [ ]:
predictions_qwen = [
    "polyp",
    "no polyp",
    "ulcer",
    "normal mucosa",
    "inflammation"
]

references_qwen = [
    ["polyp"],
    ["no polyp"],
    ["ulcer"],
    ["normal mucosa"],
    ["inflammation"]
]

In [ ]:
qwen_mean_bleu, qwen_bleu_list = mean_sentence_bleu(predictions_qwen, references_qwen)
qwen_corpus_bleu, qwen_corpus_details = corpus_bleu(predictions_qwen, references_qwen)

print("Qwen mean sentence BLEU:", qwen_mean_bleu)
print("Qwen corpus BLEU:", qwen_corpus_bleu)

Qwen mean sentence BLEU: 0.2331878710090706
Qwen corpus BLEU: 1.0
